In [1]:
import os
import zipfile

# Input folder where your downloaded zip files are stored
input_folder = r"C:\Users\Yangsi\Downloads\SRTMGL1_003-20260623_135247"

# Output folder where unzipped .hgt files will be saved
output_folder = r"C:\Users\Yangsi\Desktop\srtm_tiles"
os.makedirs(output_folder, exist_ok=True)

# Loop through all zip files in the folder
for filename in os.listdir(input_folder):
    if filename.endswith(".zip"):
        zip_path = os.path.join(input_folder, filename)
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(output_folder)
            print(f"Extracted: {filename}")
        except Exception as e:
            print(f"Failed to extract {filename}: {e}")

print("✅ All files extracted to:", output_folder)


Extracted: N01E011.SRTMGL1.hgt.zip
Extracted: N01E012.SRTMGL1.hgt.zip
Extracted: N01E013.SRTMGL1.hgt.zip
Extracted: N01E014.SRTMGL1.hgt.zip
Extracted: N01E015.SRTMGL1.hgt.zip
Extracted: N01E016.SRTMGL1.hgt.zip
Extracted: N02E009.SRTMGL1.hgt.zip
Extracted: N02E010.SRTMGL1.hgt.zip
Extracted: N02E011.SRTMGL1.hgt.zip
Extracted: N02E012.SRTMGL1.hgt.zip
Extracted: N02E013.SRTMGL1.hgt.zip
Extracted: N02E014.SRTMGL1.hgt.zip
Extracted: N02E015.SRTMGL1.hgt.zip
Extracted: N02E016.SRTMGL1.hgt.zip
Extracted: N03E008.SRTMGL1.hgt.zip
Extracted: N03E009.SRTMGL1.hgt.zip
Extracted: N03E010.SRTMGL1.hgt.zip
Extracted: N03E011.SRTMGL1.hgt.zip
Extracted: N03E012.SRTMGL1.hgt.zip
Extracted: N03E013.SRTMGL1.hgt.zip
Extracted: N03E014.SRTMGL1.hgt.zip
Extracted: N03E015.SRTMGL1.hgt.zip
Extracted: N03E016.SRTMGL1.hgt.zip
Extracted: N04E008.SRTMGL1.hgt.zip
Extracted: N04E009.SRTMGL1.hgt.zip
Extracted: N04E010.SRTMGL1.hgt.zip
Extracted: N04E011.SRTMGL1.hgt.zip
Extracted: N04E012.SRTMGL1.hgt.zip
Extracted: N04E013.S

In [3]:
import os
import rasterio
from rasterio.merge import merge
from rasterio.plot import show

# Folder containing all your .hgt files
input_folder = r"C:\Users\Yangsi\Desktop\srtm_tiles"

# Collect all .hgt files
hgt_files = [os.path.join(input_folder, f) for f in os.listdir(input_folder) if f.endswith(".hgt")]

# Open all files as raster datasets
src_files_to_mosaic = [rasterio.open(fp) for fp in hgt_files]

# Merge into one mosaic
mosaic, out_trans = merge(src_files_to_mosaic)

# Copy metadata from one of the source files
out_meta = src_files_to_mosaic[0].meta.copy()

# Update metadata for the mosaic
out_meta.update({
    "driver": "GTiff",
    "height": mosaic.shape[1],
    "width": mosaic.shape[2],
    "transform": out_trans,
    "crs": src_files_to_mosaic[0].crs
})

# Save merged DEM
output_file = r"C:\Users\Yangsi\Desktop\merged_dem.tif"
with rasterio.open(output_file, "w", **out_meta) as dest:
    dest.write(mosaic)

print("✅ Merged DEM saved as:", output_file)


✅ Merged DEM saved as: C:\Users\Yangsi\Desktop\merged_dem.tif


In [6]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np

# File paths (all in project folder)
dem_path = "merged_dem.tif"
boundary_path = "cmr_admin1.geojson"
clipped_dem_path = "cameroon_dem.tif"
slope_path = "cameroon_slope.tif"

# Step 1: Clip DEM to Cameroon boundary
boundary = gpd.read_file(boundary_path)

with rasterio.open(dem_path) as src:
    if boundary.crs != src.crs:
        boundary = boundary.to_crs(src.crs)

    out_image, out_transform = mask(src, boundary.geometry, crop=True)
    out_meta = src.meta.copy()

out_meta.update({
    "driver": "GTiff",
    "height": out_image.shape[1],
    "width": out_image.shape[2],
    "transform": out_transform
})

with rasterio.open(clipped_dem_path, "w", **out_meta) as dest:
    dest.write(out_image)

print("✅ Clipped DEM saved as:", clipped_dem_path)


✅ Clipped DEM saved as: cameroon_dem.tif


In [8]:
import rasterio
import numpy as np

dem_path = "cameroon_dem.tif"
slope_path = "cameroon_slope.tif"

with rasterio.open(dem_path) as src:
    profile = src.profile
    profile.update(dtype=rasterio.float32)

    with rasterio.open(slope_path, "w", **profile) as dst:
        for ji, window in src.block_windows(1):
            dem_block = src.read(1, window=window).astype(float)

            # Skip empty or tiny blocks
            if dem_block.size < 3 or np.all(dem_block == src.nodata):
                slope_block = np.full_like(dem_block, src.nodata, dtype=np.float32)
            else:
                dx = src.transform[0]
                dy = -src.transform[4]

                # Handle small blocks safely
                try:
                    dzdx = np.gradient(dem_block, axis=1) / dx
                    dzdy = np.gradient(dem_block, axis=0) / dy

                    slope_block = np.arctan(np.sqrt(dzdx**2 + dzdy**2)) * 180/np.pi
                    slope_block = slope_block.astype(np.float32)
                except Exception:
                    slope_block = np.full_like(dem_block, src.nodata, dtype=np.float32)

            dst.write(slope_block, 1, window=window)

print("✅ Slope raster saved as:", slope_path)


✅ Slope raster saved as: cameroon_slope.tif


In [12]:
import geopandas as gpd
import rasterstats
import pandas as pd

boundary_path = "cmr_admin1.geojson"
dem_path = "cameroon_dem.tif"
slope_path = "cameroon_slope.tif"
csv_output = "cameroon_region_stats.csv"

# Load regions
regions = gpd.read_file(boundary_path)

# Check columns
print(regions.columns)

# Use the correct region column
region_col = "adm1_name"   # change to "adm1_name1" if your file uses French names

# Compute zonal stats for elevation
elev_stats = rasterstats.zonal_stats(
    regions,
    dem_path,
    stats=["mean", "min", "max"],
    nodata=-32768
)

# Compute zonal stats for slope
slope_stats = rasterstats.zonal_stats(
    regions,
    slope_path,
    stats=["mean", "min", "max"],
    nodata=-32768
)

# Add results
regions["elevation_mean"] = [s["mean"] for s in elev_stats]
regions["elevation_min"] = [s["min"] for s in elev_stats]
regions["elevation_max"] = [s["max"] for s in elev_stats]

regions["slope_mean"] = [s["mean"] for s in slope_stats]
regions["slope_min"] = [s["min"] for s in slope_stats]
regions["slope_max"] = [s["max"] for s in slope_stats]

# Save to CSV
regions[[
    region_col,
    "elevation_mean",
    "elevation_min",
    "elevation_max",
    "slope_mean",
    "slope_min",
    "slope_max"
]].to_csv(csv_output, index=False)

print("Regional stats saved to:", csv_output)

Index(['adm1_name', 'adm1_name1', 'adm1_name2', 'adm1_name3', 'adm1_pcode',
       'adm0_name', 'adm0_name1', 'adm0_name2', 'adm0_name3', 'adm0_pcode',
       'valid_on', 'valid_to', 'area_sqkm', 'version', 'lang', 'lang1',
       'lang2', 'lang3', 'adm1_ref_name1', 'center_lat', 'center_lon',
       'geometry'],
      dtype='object')
Regional stats saved to: cameroon_region_stats.csv


In [15]:
import geopandas as gpd
import rasterstats
import pandas as pd

boundary_path = "cmr_admin1.geojson"
dem_path = "cameroon_dem.tif"
slope_path = "cameroon_slope.tif"
csv_output = "cameroon_region_stats.csv"

# Load regions
regions = gpd.read_file(boundary_path)

# Use the correct region column
region_col = "adm1_name"   # matches your elevation CSV

# Compute zonal stats for elevation
elev_stats = rasterstats.zonal_stats(
    regions,
    dem_path,
    stats=["mean", "min", "max"],
    nodata=-32768
)

# Compute zonal stats for slope
slope_stats = rasterstats.zonal_stats(
    regions,
    slope_path,
    stats=["mean", "min", "max"],
    nodata=-32768
)

# Add results
regions["elevation_mean"] = [s["mean"] for s in elev_stats]
regions["elevation_min"] = [s["min"] for s in elev_stats]
regions["elevation_max"] = [s["max"] for s in elev_stats]

regions["slope_mean"] = [s["mean"] for s in slope_stats]
regions["slope_min"] = [s["min"] for s in slope_stats]
regions["slope_max"] = [s["max"] for s in slope_stats]

# Save to CSV
regions[[
    region_col,
    "elevation_mean",
    "elevation_min",
    "elevation_max",
    "slope_mean",
    "slope_min",
    "slope_max"
]].to_csv(csv_output, index=False)

print("✅ Regional stats saved to:", csv_output)


✅ Regional stats saved to: cameroon_region_stats.csv


In [22]:
import rasterio
import numpy as np

dem_path = "cameroon_dem.tif"
slope_path = "cameroon_slope_fixed.tif"

with rasterio.open(dem_path) as src:
    profile = src.profile
    profile.update(dtype=rasterio.float32, nodata=-32768)

    with rasterio.open(slope_path, "w", **profile) as dst:
        for ji, window in src.block_windows(1):
            dem_block = src.read(1, window=window).astype(float)

            # Skip empty blocks
            if np.all(dem_block == src.nodata) or dem_block.size < 3:
                slope_block = np.full_like(dem_block, -32768, dtype=np.float32)
            else:
                dx = src.transform[0]
                dy = -src.transform[4]

                try:
                    dzdx = np.gradient(dem_block, axis=1) / dx
                    dzdy = np.gradient(dem_block, axis=0) / dy
                    slope_block = np.arctan(np.sqrt(dzdx**2 + dzdy**2)) * 180/np.pi
                    slope_block = slope_block.astype(np.float32)
                except Exception:
                    slope_block = np.full_like(dem_block, -32768, dtype=np.float32)

            dst.write(slope_block, 1, window=window)

print("✅ Fixed slope raster saved as:", slope_path)


✅ Fixed slope raster saved as: cameroon_slope_fixed.tif


In [23]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
import numpy as np
import pandas as pd

boundary_path = "cmr_admin1.geojson"
slope_path = "cameroon_slope_fixed.tif"

# Load regions
regions = gpd.read_file(boundary_path)

results = []

with rasterio.open(slope_path) as src:
    for idx, row in regions.iterrows():
        geom = [row.geometry]
        out_image, _ = mask(src, geom, crop=True)
        data = out_image[0]
        data = data[data != src.nodata]  # filter nodata

        if data.size > 0:
            slope_mean = float(np.mean(data))
            slope_min = float(np.min(data))
            slope_max = float(np.max(data))
        else:
            slope_mean = slope_min = slope_max = None

        results.append({
            "adm1_name": row["adm1_name"],  # use the same field as your elevation CSV
            "slope_mean": slope_mean,
            "slope_min": slope_min,
            "slope_max": slope_max
        })

df_slope = pd.DataFrame(results)
df_slope.to_csv("cameroon_slope_stats.csv", index=False)

print("✅ Regional slope stats saved to: cameroon_slope_stats.csv")


✅ Regional slope stats saved to: cameroon_slope_stats.csv


In [24]:
import rasterio
import numpy as np

with rasterio.open("cameroon_slope.tif") as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("NoData:", src.nodata)
    print("Count:", src.count)

    data = src.read(1)

    print("Raw min:", np.nanmin(data))
    print("Raw max:", np.nanmax(data))

    if src.nodata is not None:
        valid = data[data != src.nodata]
    else:
        valid = data[~np.isnan(data)]

    print("Valid pixels:", len(valid))

    if len(valid) > 0:
        print("Valid min:", np.nanmin(valid))
        print("Valid max:", np.nanmax(valid))
        print("Valid mean:", np.nanmean(valid))

CRS: EPSG:4326
Bounds: BoundingBox(left=8.498750000000001, bottom=1.6545833333333348, right=16.19236111111111, top=13.083472222222223)
NoData: -32768.0
Count: 1
Raw min: -32768.0
Raw max: -32768.0
Valid pixels: 0


In [25]:
import rasterio
import numpy as np

dem_path = "cameroon_dem.tif"
slope_output = "cameroon_slope_fixed.tif"

with rasterio.open(dem_path) as src:
    dem = src.read(1).astype("float64")
    profile = src.profile.copy()
    nodata = src.nodata
    transform = src.transform

    if nodata is not None:
        dem[dem == nodata] = np.nan

    # Convert pixel size from degrees to meters approximately
    xres = abs(transform.a) * 111320
    yres = abs(transform.e) * 111320

    # Calculate elevation change in x and y directions
    dz_dy, dz_dx = np.gradient(dem, yres, xres)

    # Calculate slope in degrees
    slope_rad = np.arctan(np.sqrt(dz_dx**2 + dz_dy**2))
    slope_deg = np.degrees(slope_rad)

    # Set NaN to NoData
    slope_deg[np.isnan(slope_deg)] = -9999

    profile.update(
        dtype=rasterio.float32,
        nodata=-9999,
        count=1
    )

    with rasterio.open(slope_output, "w", **profile) as dst:
        dst.write(slope_deg.astype(rasterio.float32), 1)

print("Saved:", slope_output)

MemoryError: Unable to allocate 8.49 GiB for an array with shape (41142, 27697) and data type float64